# 02 · Feature Engineering
**Payment Failure Intelligence & Revenue Optimization System**

**Goal:** Create meaningful derived features from cleaned data for analysis and modelling.

In [ ]:
import pandas as pd
import numpy as np
import os

os.makedirs('../outputs', exist_ok=True)

df = pd.read_csv('../outputs/cleaned_data.csv', parse_dates=['timestamp'])
print(f"Loaded cleaned data: {df.shape}")

## A · Time Features

In [ ]:
df['hour']        = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek          # 0=Mon, 6=Sun
df['day_name']    = df['timestamp'].dt.day_name()
df['month']       = df['timestamp'].dt.month
df['month_name']  = df['timestamp'].dt.strftime('%b')
df['year']        = df['timestamp'].dt.year
df['date']        = df['timestamp'].dt.date
df['is_weekend']  = df['day_of_week'].isin([5, 6]).astype(int)
df['is_night']    = ((df['hour'] >= 0) & (df['hour'] < 6)).astype(int)
df['time_slot']   = pd.cut(
    df['hour'],
    bins=[-1, 5, 11, 16, 20, 23],
    labels=['Night (0–5)', 'Morning (6–11)', 'Afternoon (12–16)',
            'Evening (17–20)', 'Late Eve (21–23)']
)

print("Time features added:")
print(df[['timestamp', 'hour', 'day_name', 'is_weekend', 'is_night', 'time_slot']].head(5))

## B · Transaction Features

In [ ]:
df['log_amount']    = np.log1p(df['amount'])
df['amount_bucket'] = pd.cut(
    df['amount'],
    bins=[0, 1_000, 10_000, 50_000, np.inf],
    labels=['Low (<₹1K)', 'Medium (₹1K–10K)', 'High (₹10K–50K)', 'Very High (>₹50K)']
)
df['is_failed'] = (df['status'] == 'Failed').astype(int)

print("\nTransaction features:")
print(df[['amount', 'log_amount', 'amount_bucket', 'is_failed']].describe())

## C · User Features

In [ ]:
user_stats = df.groupby('user_id').agg(
    transactions_per_user=('transaction_id', 'count'),
    user_total_amount    =('amount',         'sum'),
    user_failure_rate    =('is_failed',      'mean'),
).reset_index()

user_stats['user_type'] = np.where(
    user_stats['transactions_per_user'] >= 15, 'Returning', 'New'
)

df = df.merge(user_stats[['user_id', 'transactions_per_user',
                           'user_failure_rate', 'user_type']],
              on='user_id', how='left')

print("\nUser features sample:")
print(df[['user_id', 'transactions_per_user', 'user_failure_rate', 'user_type']].head(5))
print("\nUser type distribution:")
print(df['user_type'].value_counts())

## D · Rolling Features (7-day failure rate)

In [ ]:
daily = (df.groupby('date')
           .agg(total=('transaction_id', 'count'), failed=('is_failed', 'sum'))
           .reset_index())
daily['date']             = pd.to_datetime(daily['date'])
daily['daily_fail_rate']  = daily['failed'] / daily['total']
daily['rolling_7d_rate']  = daily['daily_fail_rate'].rolling(7, min_periods=1).mean()

print("\nRolling 7-day failure rate sample:")
print(daily.tail(10))

## E · Export Engineered Data

In [ ]:
df.to_csv('../outputs/engineered_data.csv', index=False)
daily.to_csv('../outputs/daily_trends.csv', index=False)
print(f"\n✓ Saved engineered_data.csv  ({len(df):,} rows, {df.shape[1]} columns)")
print(f"✓ Saved daily_trends.csv     ({len(daily):,} days)")

print("\nFinal columns:")
print(list(df.columns))
print("\n=== FEATURE ENGINEERING COMPLETE ===")